# Анализ ошибок и важность признаков

In [1]:
import pandas as pd
import numpy as np
from scipy.sparse import hstack, csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, precision_recall_curve
from features import build_features

df = pd.read_csv('dataset.csv')
df = build_features(df)
df = df.dropna(subset=['дата_публикации']).sort_values('дата_публикации').reset_index(drop=True)

split_idx = int(len(df) * 0.8)
train = df.iloc[:split_idx]
test = df.iloc[split_idx:].reset_index(drop=True)
print(len(train), len(test))

1693 424


In [2]:
tfidf = TfidfVectorizer(max_features=200)
X_train_text = tfidf.fit_transform(train['название_стем'])
X_test_text = tfidf.transform(test['название_стем'])

cat_cols = ['способ_группа', 'тип_торгов']
ohe = OneHotEncoder(handle_unknown='ignore')
X_train_cat = ohe.fit_transform(train[cat_cols])
X_test_cat = ohe.transform(test[cat_cols])

num_cols = ['москва', 'нмц_указана', 'лог_цена', 'дни_до_дедлайна']
X_train_num = csr_matrix(train[num_cols].values)
X_test_num = csr_matrix(test[num_cols].values)

X_train = hstack([X_train_text, X_train_cat, X_train_num])
X_test = hstack([X_test_text, X_test_cat, X_test_num])

feature_names = np.array(list(tfidf.get_feature_names_out()) + list(ohe.get_feature_names_out(cat_cols)) + num_cols)

y_train = train['label']
y_test = test['label']

rf = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)

proba = rf.predict_proba(X_test)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_test, proba)
target_recall = 0.85
idx = np.where(recall[:-1] >= target_recall)[0]
best = idx[np.argmax(precision[idx])] if len(idx) else np.argmax(recall[:-1])
threshold = thresholds[best]
print(f"Порог: {threshold:.3f}, precision={precision[best]:.3f}, recall={recall[best]:.3f}")

test = test.copy()
test['скор'] = proba
test['предсказание'] = (proba >= threshold).astype(int)

Порог: 0.107, precision=0.411, recall=0.920


## Важность признаков

In [3]:
importances = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=False)
importances.head(20)

дни_до_дедлайна                 0.172262
лог_цена                        0.070518
способ_группа_Прочее            0.049417
тип_торгов_Коммерческие         0.031912
тип_торгов_Малые закупки        0.031275
програ                          0.027483
лиценз                          0.021910
нмц_указана                     0.019879
систем                          0.019569
исполь                          0.019112
разраб                          0.018212
постав                          0.016375
развит                          0.015799
способ_группа_Тендер/Конкурс    0.015274
предос                          0.015267
информ                          0.014221
обеспе                          0.014096
внедре                          0.011901
москва                          0.011841
корпор                          0.010680
dtype: float64

## Ложноотрицательные

In [4]:
fn = test[(test['label']==1) & (test['предсказание']==0)]
print(f"Пропущено {len(fn)} из {test['label'].sum()} интересных")
fn[['Название', 'способ_группа', 'тип_торгов', 'москва', 'лог_цена', 'скор']].sort_values('скор', ascending=False)

Пропущено 4 из 50 интересных


,Название,способ_группа,тип_торгов,москва,лог_цена,скор
322,Запрос предложений в электронной форме на прав...,Запрос предложений,Коммерческие,1,14.382623,0.096667
347,Выполнение работ по доработке Внешнего портала...,Запрос предложений,Коммерческие,1,0.000000,0.076667
20,Оказание услуг по архитектурному сопровождению...,Тендер/Конкурс,Коммерческие,1,13.604791,0.043333
184,"Оказание услуг (выполнение работ) в объеме, пр...",Запрос цен/котировок,223-ФЗ,0,15.448808,0.026667


## Ложноположительные

In [5]:
fp = test[(test['label']==0) & (test['предсказание']==1)]
print(f"Ложных срабатываний: {len(fp)} из {len(test) - test['label'].sum()} неинтересных")
fp[['Название', 'способ_группа', 'тип_торгов', 'москва', 'лог_цена', 'скор']].sort_values('скор', ascending=False).head(15)

Ложных срабатываний: 66 из 374 неинтересных


,Название,способ_группа,тип_торгов,москва,лог_цена,скор
113,Конкурс в электронной форме участниками которо...,Тендер/Конкурс,223-ФЗ,1,16.722412,0.703333
257,Выполнение поручений по разработке и сопровожд...,Тендер/Конкурс,223-ФЗ,1,15.872434,0.666667
359,Эшелонированная защита веб-сайтов и приложений...,Запрос предложений,Коммерческие,1,0.000000,0.600000
126,Конкурс в электронной форме участниками которо...,Тендер/Конкурс,223-ФЗ,1,15.522223,0.553333
79,"Оказание услуг по заведению, поддержке и запол...",Тендер/Конкурс,Коммерческие,1,0.000000,0.503333
46,Оказание услуг по технической поддержке СЗИ Ch...,Запрос предложений,Коммерческие,1,0.000000,0.493333
267,"Разработка корпоративного сайта ООО ""РНК""",Запрос предложений,Коммерческие,0,0.000000,0.480000
389,Выполнение работ по развитию и сопровождению М...,Запрос предложений,223-ФЗ,1,15.894952,0.466667
24,"Выполнение работ по развитию АИС ""Взаимодейств...",Запрос предложений,Коммерческие,1,12.948012,0.393333
333,№KFFTGO_02_Оказание услуг по оценке текущего с...,Прочее,Коммерческие,0,0.000000,0.390000
